In [8]:
from deepagent_lang import E2BCodeRunner, build_e2b_code_execution_tool
import json
# 2. Basic execution

runner = E2BCodeRunner()

result = runner.execute(
    code="""
x = 20
y = 22
final_answer = x + y
final_answer
"""
)

print(result)
# print(json.loads(result))
# print("FINAL RESULT:", result["main_result"])
#convert json into dict
result_dict = json.loads(result)
print("FINAL RESULT:", result_dict["main_result"])

{
  "status": "ok",
  "sandbox_id": "iccc0qwq21ph1djzuzgto",
  "context_id": "b2870e72-82c8-457c-a3a8-0b4e70987974",
  "uploaded_files": [],
  "installed_python_packages": [],
  "executed_code": "x = 20\ny = 22\nfinal_answer = x + y\nfinal_answer",
  "main_result": "42",
  "stdout": [],
  "stderr": [],
  "result_formats": [
    [
      "text"
    ]
  ],
  "error": null
}
FINAL RESULT: 42


3. Structured error/log output

  result = runner.execute(
      code="""
  print("starting")
  1 / 0
  """
  )

  parsed = json.loads(result)
  print(json.dumps(parsed, indent=2))

  Expected:

  - status = "error"
  - stdout contains "starting"
  - error["name"] should be something like ZeroDivisionError

In [9]:
result = runner.execute(
      code="""
  print("starting")
  1 / 0
  """
  )

parsed = json.loads(result)
print(json.dumps(parsed, indent=2))

{
  "status": "error",
  "sandbox_id": "iccc0qwq21ph1djzuzgto",
  "context_id": "b2870e72-82c8-457c-a3a8-0b4e70987974",
  "uploaded_files": [],
  "installed_python_packages": [],
  "executed_code": "print(\"starting\")\n  1 / 0",
  "main_result": null,
  "stdout": [],
  "stderr": [],
  "result_formats": [],
  "error": {
    "name": "IndentationError",
    "value": "unexpected indent (2823363537.py, line 2)",
    "traceback": "  Cell In[2], line 2\n    1 / 0\n    ^\nIndentationError: unexpected indent\n"
  }
}


In [10]:
#   4. File upload test

#   This verifies the useful extra your simple function does not have.

result = runner.execute(
    code="""
import pandas as pd

df = pd.read_csv("/home/user/input/01_expenses.csv")
final_answer = {
    "rows": len(df),
    "columns": list(df.columns),
}
final_answer
""",
    files=["sample_data/expenses.csv"],
    python_packages=["pandas"],
)

parsed = json.loads(result)
print(json.dumps(parsed, indent=2))

# Expected:

# - status = "ok"
# - uploaded_files should show your local CSV and remote sandbox path
# - main_result should contain row count / columns

{
  "status": "ok",
  "sandbox_id": "iccc0qwq21ph1djzuzgto",
  "context_id": "b2870e72-82c8-457c-a3a8-0b4e70987974",
  "uploaded_files": [
    {
      "local_path": "C:\\Users\\Sarthak Kaushik\\Desktop\\Playground2\\Deepagent_lang\\sample_data\\expenses.csv",
      "remote_path": "/home/user/input/01_expenses.csv"
    }
  ],
  "installed_python_packages": [
    "pandas"
  ],
  "executed_code": "import pandas as pd\n\ndf = pd.read_csv(\"/home/user/input/01_expenses.csv\")\nfinal_answer = {\n    \"rows\": len(df),\n    \"columns\": list(df.columns),\n}\nfinal_answer",
  "main_result": "{'rows': 37,\n 'columns': ['expense_id',\n  'date_incurred',\n  'logged_by',\n  'belongs_to',\n  'status',\n  'category',\n  'subcategory',\n  'description',\n  'merchant_or_item',\n  'amount',\n  'currency',\n  'is_recurring',\n  'confidence',\n  'created_at',\n  'updated_at']}",
  "stdout": [],
  "stderr": [],
  "result_formats": [
    [
      "text",
      "json"
    ]
  ],
  "error": null
}


### Step-1

In [1]:
import os
import sys
from pathlib import Path

repo_root = Path(r"C:\Users\Sarthak Kaushik\Desktop\Playground2\Deepagent_lang")
if str(repo_root) not in sys.path:
    sys.path.append(str(repo_root))

repo_root = Path(r"C:\Users\Sarthak Kaushik\Desktop\Playground2\Deepagent_lang")
if str(repo_root) not in sys.path:
    sys.path.append(str(repo_root))
repo_root

WindowsPath('C:/Users/Sarthak Kaushik/Desktop/Playground2/Deepagent_lang')

In [25]:
from deepagents import create_deep_agent
from langchain.chat_models import init_chat_model
from deepagent_lang import build_e2b_code_execution_tool
from langchain.chat_models import init_chat_model
import json
# 3. Build the tool
tool = build_e2b_code_execution_tool(workspace_root=repo_root)
# python_tool = build_e2b_code_execution_tool((workspace_root=repo_root))
# python_tool

# 4. Build the agent
    
agent = create_deep_agent(
      model=init_chat_model("openai:gpt-5.3-codex"),
      tools=[tool],
      system_prompt=(
          "You are reserch agent with access to a Python environment where you can run code and upload files. "
          "For every question, think step by step and decide if you need to run code to answer it. "
          "Creat a proper step by step plan to answer the question. If you need to run code, use the tool `execute_python_in_e2b` with the code you want to run and any files you want to upload. "
          "You must use `execute_python_in_e2b` for CSV or file analysis. "
          "When a user provides a workspace file path, pass it in the tool's `files` argument. "
          "After upload, do not read the local path in Python. "
          "For the first uploaded file, read `/home/user/input/01_expenses.csv`. "
          "Use pandas when helpful."
      ),
  )

In [26]:
# result = tool.invoke({
#     "code": """
# import pandas as pd

# df = pd.read_csv("/home/user/input/01_expenses.csv")
# final_answer = {
#     "row_count": len(df),
#     "columns": list(df.columns),
# }
# final_answer
# """,
#     "files": ["sample_data/expenses.csv"],
#     "python_packages": ["pandas"],
# })
# result

In [27]:
result=tool.invoke({
    'code': """import pandas as pd
\n\n
path = '/home/user/input/expenses.csv'\n
df = pd.read_csv(path)\n
print('shape', df.shape)\n
print('columns', list(df.columns))\n
print(df.head(10).to_string(index=False))\n
print('\\ninfo:')\nprint(df.dtypes)\n"""})
result

'{\n  "status": "error",\n  "sandbox_id": "i88brg3nxfvl8g2ftqnln",\n  "context_id": "4fee72e3-3227-469c-8d25-42da2f9236d8",\n  "uploaded_files": [],\n  "installed_python_packages": [],\n  "executed_code": "import pandas as pd\\n\\n\\n\\npath = \'/home/user/input/expenses.csv\'\\n\\ndf = pd.read_csv(path)\\n\\nprint(\'shape\', df.shape)\\n\\nprint(\'columns\', list(df.columns))\\n\\nprint(df.head(10).to_string(index=False))\\n\\nprint(\'\\\\ninfo:\')\\nprint(df.dtypes)",\n  "main_result": null,\n  "stdout": [],\n  "stderr": [],\n  "result_formats": [],\n  "error": {\n    "name": "FileNotFoundError",\n    "value": "[Errno 2] No such file or directory: \'/home/user/input/expenses.csv\'",\n    "traceback": "---------------------------------------------------------------------------FileNotFoundError                         Traceback (most recent call last)Cell In[1], line 12\\n      6 import pandas as pd\\n     10 path = \'/home/user/input/expenses.csv\'\\n---> 12 df = pd.read_csv(path)\\n 

In [28]:
# result = tool.invoke({
#     "code": """
# import pandas as pd

# df = pd.read_csv("/home/user/input/01_expenses.csv")

# cat_candidates = ["category", "Category", "cat", "expense_category"]
# amt_candidates = ["amount", "Amount", "amt", "cost", "value", "price"]

# cat_col = next((c for c in cat_candidates if c in df.columns), None)
# if cat_col is None:
#     object_cols = [c for c in df.columns if df[c].dtype == "object"]
#     cat_col = object_cols[0] if object_cols else df.columns[0]

# amt_col = next((c for c in amt_candidates if c in df.columns), None)
# if amt_col is None:
#     numeric_cols = df.select_dtypes(include=["number"]).columns.tolist()
#     amt_col = numeric_cols[0] if numeric_cols else df.columns[1]

# df[amt_col] = pd.to_numeric(df[amt_col], errors="coerce")

# final_answer = {
#     "row_count": len(df),
#     "top_5_categories": (
#         df.groupby(cat_col)[amt_col]
#         .sum()
#         .sort_values(ascending=False)
#         .head(5)
#         .to_dict()
#     ),
# }
# final_answer
# """,
#     "files": ["sample_data/expenses.csv"],
#     "python_packages": ["pandas"],
# })

# # print(json.dumps(json.loads(result), indent=2))

# parsed = json.loads(result)
# print(json.dumps(parsed, indent=2))

In [29]:
# 5. Small helper to print the final assistant message

def extract_text(content):
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        parts = []
        for item in content:
            if isinstance(item, str):
                parts.append(item)
            elif isinstance(item, dict) and item.get("type") == "text":
                parts.append(str(item.get("text", "")))
            else:
                parts.append(str(item))
        return "\n".join(part for part in parts if part).strip()
    return str(content)

In [42]:
# # 6. Run the agent on your CSV

query = "Calculate how many r's are in the word 'strawberry'?"
# query = "Analyze sample_data/expenses.csv and tell me the top 5 categories by total spend."
# query = "hi"
query= "How can you help me ?"
query= "What tools do you have access to ?"
# query = "Can you tell me how should I optimize my spends by creatin the plan and execute secquentially -  based on the analysis of the expenses.csv file, file path is - 'sample_data/expenses.csv' ?"
response = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content" :query,
            }
        ]
    }
)

In [43]:
#Prin the Todo of the Agent
print(response.get("todos", []))

[]


In [44]:
# 7. Print the final answer

print(extract_text(response["messages"][-1].content))

- `execute_python_in_e2b` — run Python code in a sandbox (with optional file uploads and pip packages).
- Filesystem tools:
  - `ls` — list directory contents
  - `read_file` — read files
  - `write_file` — create/write files
  - `edit_file` — exact string replacement edits
  - `glob` — find files by pattern
  - `grep` — search text across files
- `write_todos` — create/update a structured task list for multi-step work.
- `task` — spawn a short-lived subagent (`general-purpose`) for isolated complex tasks.
- `multi_tool_use.parallel` — run independent tool calls in parallel.


In [33]:
# 4. If the agent still fails, inspect whether it actually called the tool

for i, message in enumerate(response["messages"], start=1):
    print(f"\n--- {i}: {type(message).__name__} ---")
    if hasattr(message, "tool_calls"):
        print("tool_calls:", message.tool_calls)
    if hasattr(message, "content"):
        print(extract_text(message.content))


--- 1: HumanMessage ---
Can you tell me how should I optimize my spends by creatin the plan and execute secquentially -  based on the analysis of the expenses.csv file, file path is - 'sample_data/expenses.csv' ?

--- 2: AIMessage ---
tool_calls: [{'name': 'execute_python_in_e2b', 'args': {'code': 'import pandas as pd\nimport numpy as np\n\n# Read uploaded file from required sandbox path\npath = \'/home/user/input/01_expenses.csv\'\ndf = pd.read_csv(path)\n\n# Basic cleaning\ncols = {c:c.strip() for c in df.columns}\ndf = df.rename(columns=cols)\n\n# Try to detect key columns\nlower_map = {c.lower(): c for c in df.columns}\n\n# category column\ncat_col = None\nfor cand in [\'category\',\'expense_category\',\'type\',\'merchant_category\',\'group\']:\n    if cand in lower_map:\n        cat_col = lower_map[cand]\n        break\n\n# amount column\namt_col = None\nfor cand in [\'amount\',\'expense\',\'spent\',\'cost\',\'value\']:\n    if cand in lower_map:\n        amt_col = lower_map[cand